In [9]:
import os
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_0ffe570bb0120a8a77d3e4011c945192'

import kagglehub
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Path:", path)
for f in os.listdir(path):
    print(f)

Path: /Users/Tanya/.cache/kagglehub/datasets/olistbr/brazilian-ecommerce/versions/2
olist_sellers_dataset.csv
product_category_name_translation.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv


In [10]:
import pandas as pd

orders      = pd.read_csv(f"{path}/olist_orders_dataset.csv")
order_items = pd.read_csv(f"{path}/olist_order_items_dataset.csv")
payments    = pd.read_csv(f"{path}/olist_order_payments_dataset.csv")
customers   = pd.read_csv(f"{path}/olist_customers_dataset.csv")
products    = pd.read_csv(f"{path}/olist_products_dataset.csv")
sellers     = pd.read_csv(f"{path}/olist_sellers_dataset.csv")
reviews     = pd.read_csv(f"{path}/olist_order_reviews_dataset.csv")
translation = pd.read_csv(f"{path}/product_category_name_translation.csv")

for name, df in zip(
    ['orders','items','payments','customers','products','sellers','reviews','translation'],
    [orders, order_items, payments, customers, products, sellers, reviews, translation]
):
    print(f"{name}: {df.shape}")

orders: (99441, 8)
items: (112650, 7)
payments: (103886, 5)
customers: (99441, 5)
products: (32951, 9)
sellers: (3095, 4)
reviews: (99224, 7)
translation: (71, 2)


In [11]:
# See column names of each table
for name, df in zip(
    ['orders','items','payments','customers','products','sellers','reviews'],
    [orders, order_items, payments, customers, products, sellers, reviews]
):
    print(f"\n--- {name} ---")
    print(df.columns.tolist())


--- orders ---
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

--- items ---
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

--- payments ---
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

--- customers ---
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

--- products ---
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

--- sellers ---
['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

--- reviews ---
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'revie

In [12]:
# Check nulls in each table
for name, df in zip(
    ['orders','items','payments','customers','products','sellers','reviews'],
    [orders, order_items, payments, customers, products, sellers, reviews]
):
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]  # only show columns that have nulls
    if len(nulls) > 0:
        print(f"\n--- {name} ---")
        print(nulls)
    else:
        print(f"\n--- {name} --- no nulls")


--- orders ---
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

--- items --- no nulls

--- payments --- no nulls

--- customers --- no nulls

--- products ---
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

--- sellers --- no nulls

--- reviews ---
review_comment_title      87656
review_comment_message    58247
dtype: int64


In [13]:
# --- ORDERS ---
# Keep only delivered orders
orders = orders[orders['order_status'] == 'delivered'].copy()

# Convert date columns to datetime
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

# Engineer delivery delay
orders['delivery_delay_days'] = (
    orders['order_delivered_customer_date'] - 
    orders['order_estimated_delivery_date']
).dt.days

# is_late flag
orders['is_late'] = orders['delivery_delay_days'] > 0

print(f"Orders after filtering to delivered: {orders.shape}")
print(f"Late orders: {orders['is_late'].sum()} ({orders['is_late'].mean()*100:.1f}%)")

# --- PRODUCTS ---
# Merge english category names
products = products.merge(translation, on='product_category_name', how='left')
products['product_category_name_english'] = products['product_category_name_english'].fillna('unknown')

# Drop rows missing dimensions
products = products.dropna(subset=['product_weight_g'])
print(f"\nProducts after cleaning: {products.shape}")

# --- REVIEWS ---
# Keep only what we need
reviews_clean = reviews[['order_id', 'review_score']].copy()
print(f"Reviews clean: {reviews_clean.shape}")

Orders after filtering to delivered: (96478, 10)
Late orders: 6534 (6.8%)

Products after cleaning: (32949, 10)
Reviews clean: (99224, 2)


In [14]:
# Step 1 — aggregate items per order (price + freight)
items_agg = order_items.groupby('order_id').agg(
    order_value        = ('price', 'sum'),
    freight_value      = ('freight_value', 'sum'),
    item_count         = ('order_item_id', 'count'),
    product_id         = ('product_id', 'first'),  # primary product
    seller_id          = ('seller_id', 'first')
).reset_index()

# Step 2 — aggregate payments per order
payments_agg = payments.groupby('order_id').agg(
    payment_value      = ('payment_value', 'sum'),
    payment_type       = ('payment_type', 'first'),
    payment_installments = ('payment_installments', 'max')
).reset_index()

# Step 3 — join everything onto orders
fact_transactions = orders.merge(customers, on='customer_id', how='left')
fact_transactions = fact_transactions.merge(items_agg, on='order_id', how='left')
fact_transactions = fact_transactions.merge(payments_agg, on='order_id', how='left')
fact_transactions = fact_transactions.merge(reviews_clean, on='order_id', how='left')
fact_transactions = fact_transactions.merge(
    products[['product_id', 'product_category_name_english']], 
    on='product_id', how='left'
)

# Step 4 — add time dimensions
fact_transactions['order_year']    = fact_transactions['order_purchase_timestamp'].dt.year
fact_transactions['order_month']   = fact_transactions['order_purchase_timestamp'].dt.month
fact_transactions['order_quarter'] = fact_transactions['order_purchase_timestamp'].dt.quarter
fact_transactions['order_dow']     = fact_transactions['order_purchase_timestamp'].dt.day_name()
fact_transactions['order_week']    = fact_transactions['order_purchase_timestamp'].dt.isocalendar().week.astype(int)

# Step 5 — total revenue including freight
fact_transactions['total_revenue'] = fact_transactions['order_value'] + fact_transactions['freight_value']

print(f"Master table shape: {fact_transactions.shape}")
print(f"\nColumns: {fact_transactions.columns.tolist()}")
print(f"\nNull check:")
print(fact_transactions.isnull().sum()[fact_transactions.isnull().sum() > 0])

Master table shape: (97007, 30)

Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_delay_days', 'is_late', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_value', 'freight_value', 'item_count', 'product_id', 'seller_id', 'payment_value', 'payment_type', 'payment_installments', 'review_score', 'product_category_name_english', 'order_year', 'order_month', 'order_quarter', 'order_dow', 'order_week', 'total_revenue']

Null check:
order_approved_at                 14
order_delivered_carrier_date       2
order_delivered_customer_date      8
delivery_delay_days                8
payment_value                      1
payment_type                       1
payment_installments               1
review_score                     646
product_category_name_english     16
dtype: int64


In [15]:
# Fill small nulls
fact_transactions['review_score'] = fact_transactions['review_score'].fillna(
    fact_transactions['review_score'].median()
)
fact_transactions['product_category_name_english'] = fact_transactions['product_category_name_english'].fillna('unknown')

# Drop the 1 row missing payment info — too small to matter
fact_transactions = fact_transactions.dropna(subset=['payment_value'])

# For delivery dates — fill with estimated (these are edge cases)
fact_transactions['order_delivered_customer_date'] = fact_transactions[
    'order_delivered_customer_date'].fillna(fact_transactions['order_estimated_delivery_date']
)
fact_transactions['delivery_delay_days'] = fact_transactions['delivery_delay_days'].fillna(0)

print(f"Shape after fixing nulls: {fact_transactions.shape}")
print(f"Any nulls left: {fact_transactions.isnull().sum().sum()}")

Shape after fixing nulls: (97006, 30)
Any nulls left: 16


In [16]:
# Reference date — one day after the last order in the dataset
reference_date = fact_transactions['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

rfm = fact_transactions.groupby('customer_unique_id').agg(
    recency    = ('order_purchase_timestamp', lambda x: (reference_date - x.max()).days),
    frequency  = ('order_id', 'nunique'),
    monetary   = ('total_revenue', 'sum')
).reset_index()

print(rfm.describe())
print(f"\nSample:\n{rfm.head()}")

            recency     frequency      monetary
count  93357.000000  93357.000000  93357.000000
mean     237.936673      1.033420    165.917093
std      152.584315      0.209099    227.788213
min        1.000000      1.000000      9.590000
25%      114.000000      1.000000     63.100000
50%      219.000000      1.000000    107.890000
75%      346.000000      1.000000    183.120000
max      695.000000     15.000000  13664.080000

Sample:
                 customer_unique_id  recency  frequency  monetary
0  0000366f3b9a7992bf8c76cfdf3221e2      112          1    141.90
1  0000b849f77a49e4a4ce2b2a4ca5be3f      115          1     27.19
2  0000f46a3911fa3c0805444483337064      537          1     86.22
3  0000f6ccb0745a6a4b88665a16c9f078      321          1     43.62
4  0004aac84e0df4da2b147fca70cf8255      288          1    196.89


In [17]:
# Score each dimension 1-5 using quintiles
rfm['R_score'] = pd.qcut(rfm['recency'],   q=5, labels=[5,4,3,2,1]).astype(int)
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm['M_score'] = pd.qcut(rfm['monetary'],  q=5, labels=[1,2,3,4,5]).astype(int)

# Combined RFM score
rfm['RFM_score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)

# Segment mapping
def assign_segment(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    if r >= 4 and f >= 4:
        return 'Champion'
    elif r >= 3 and f >= 3:
        return 'Loyal'
    elif r >= 4 and f <= 2:
        return 'New Customer'
    elif r >= 3 and f <= 2 and m >= 3:
        return 'Potential Loyalist'
    elif r == 2 and f >= 2:
        return 'At Risk'
    elif r <= 2 and f <= 2 and m >= 3:
        return 'Cant Lose Them'
    else:
        return 'Lost'

rfm['segment'] = rfm.apply(assign_segment, axis=1)

# Segment summary
summary = rfm.groupby('segment').agg(
    customer_count = ('customer_unique_id', 'count'),
    avg_recency    = ('recency', 'mean'),
    avg_frequency  = ('frequency', 'mean'),
    avg_monetary   = ('monetary', 'mean')
).round(1).sort_values('customer_count', ascending=False)

print(summary)
print(f"\nTotal customers: {rfm.shape[0]:,}")

                    customer_count  avg_recency  avg_frequency  avg_monetary
segment                                                                     
Lost                         18846        420.0            1.0         120.3
Loyal                        18824        168.8            1.0         162.4
New Customer                 14984         90.9            1.0         163.5
Champion                     14961         90.2            1.1         177.7
At Risk                      14921        316.5            1.0         169.8
Cant Lose Them                6467        421.3            1.0         240.0
Potential Loyalist            4354        220.7            1.0         223.2

Total customers: 93,357


In [18]:
rfm.reset_index().to_csv('dim_customers.csv', index=False)
print("✓ dim_customers.csv saved:", rfm.shape[0], "rows")
print(rfm.reset_index().columns.tolist())

✓ dim_customers.csv saved: 93357 rows
['index', 'customer_unique_id', 'recency', 'frequency', 'monetary', 'R_score', 'F_score', 'M_score', 'RFM_score', 'segment']


In [19]:
rfm_export = rfm.reset_index()  # brings customer_unique_id back as a column
rfm_export.index.name = None
rfm_export.to_csv('dim_customers.csv', index=False)
print("✓ Columns:", rfm_export.columns.tolist())

✓ Columns: ['index', 'customer_unique_id', 'recency', 'frequency', 'monetary', 'R_score', 'F_score', 'M_score', 'RFM_score', 'segment']


In [20]:
rfm_export = rfm.reset_index(drop=True)
print(rfm_export.columns.tolist())
print(rfm_export.head(2))

['customer_unique_id', 'recency', 'frequency', 'monetary', 'R_score', 'F_score', 'M_score', 'RFM_score', 'segment']
                 customer_unique_id  recency  frequency  monetary  R_score  \
0  0000366f3b9a7992bf8c76cfdf3221e2      112          1    141.90        4   
1  0000b849f77a49e4a4ce2b2a4ca5be3f      115          1     27.19        4   

   F_score  M_score RFM_score       segment  
0        1        4       414  New Customer  
1        1        1       411  New Customer  


In [21]:
rfm.to_csv('dim_customers.csv', index=False)
print("✓ dim_customers.csv saved")
print("✓ Columns:", rfm.columns.tolist())
print("✓ Rows:", len(rfm))

✓ dim_customers.csv saved
✓ Columns: ['customer_unique_id', 'recency', 'frequency', 'monetary', 'R_score', 'F_score', 'M_score', 'RFM_score', 'segment']
✓ Rows: 93357


In [22]:
import numpy as np

np.random.seed(42)

dates = pd.date_range('2017-01-01', '2018-08-31', freq='D')

# ── Channel personality (impressions per $, CTR, CVR, AOV) ─────────────────
channel_config = {
    'Instagram':  {'base_spend': 300, 'imp_per_dollar': (30, 50), 'ctr': (0.02, 0.05), 'cvr': (0.02, 0.06),  'aov': (60,  100), 'noise': 0.2},
    'Google':     {'base_spend': 500, 'imp_per_dollar': (15, 25), 'ctr': (0.05, 0.12), 'cvr': (0.08, 0.15),  'aov': (80,  130), 'noise': 0.15},
    'Email':      {'base_spend': 80,  'imp_per_dollar': (40, 70), 'ctr': (0.10, 0.25), 'cvr': (0.10, 0.20),  'aov': (50,  90),  'noise': 0.25},
    'Influencer': {'base_spend': 200, 'imp_per_dollar': (50, 80), 'ctr': (0.01, 0.04), 'cvr': (0.01, 0.04),  'aov': (40,  80),  'noise': 0.35},
}

rows = []

for date in dates:
    # Seasonality
    month_factor = 1.0
    if date.month in [11, 12]:
        month_factor = 1.6
    elif date.month in [1, 2]:
        month_factor = 0.75

    # Weekend dip
    weekend_factor = 0.7 if date.dayofweek >= 5 else 1.0

    # Campaign burst
    campaign_burst = 1.5 if np.random.rand() < 0.10 else 1.0

    for channel, cfg in channel_config.items():
        noise = 1 + np.random.uniform(-cfg['noise'], cfg['noise'])
        spend = cfg['base_spend'] * month_factor * weekend_factor * campaign_burst * noise
        spend = round(max(spend, 10), 2)

        # ── Funnel ────────────────────────────────────────────────────────
        impressions  = int(spend * np.random.uniform(*cfg['imp_per_dollar']))
        clicks       = int(impressions * np.random.uniform(*cfg['ctr']))
        conversions  = int(clicks * np.random.uniform(*cfg['cvr']))
        conversions  = max(conversions, 1)

        aov          = np.random.uniform(*cfg['aov'])
        revenue      = round(conversions * aov, 2)

        # ── Derived metrics (not inputs) ──────────────────────────────────
        roas         = round(revenue / spend, 3)
        cac          = round(spend / conversions, 2)
        ctr          = round(clicks / impressions, 4) if impressions > 0 else 0
        cvr          = round(conversions / clicks, 4) if clicks > 0 else 0

        rows.append({
            'date':               date,
            'channel':            channel,
            'spend':              spend,
            'impressions':        impressions,
            'clicks':             clicks,
            'conversions':        conversions,
            'attributed_revenue': revenue,
            'roas':               roas,
            'cac':                cac,
            'ctr':                ctr,
            'cvr':                cvr,
        })

fact_marketing_spend = pd.DataFrame(rows)

print("✓ Marketing spend table built:", fact_marketing_spend.shape)
print(fact_marketing_spend.head(8))
print("\nChannel summary:")
print(fact_marketing_spend.groupby('channel')[['spend','impressions','clicks','conversions','attributed_revenue','roas','cac']].mean().round(2))

✓ Marketing spend table built: (2432, 11)
        date     channel   spend  impressions  clicks  conversions  \
0 2017-01-01   Instagram  185.90         8298     314            8   
1 2017-01-01      Google  227.70         5387     496           64   
2 2017-01-01       Email   51.87         3370     444           52   
3 2017-01-01  Influencer   90.61         5956     136            2   
4 2017-01-02   Instagram  206.29         7700     259           13   
5 2017-01-02      Google  376.60         7880     419           51   
6 2017-01-02       Email   46.95         3214     786          142   
7 2017-01-02  Influencer  107.76         7599     176            2   

   attributed_revenue     roas    cac     ctr     cvr  
0              529.92    2.851  23.24  0.0378  0.0255  
1             5185.87   22.775   3.56  0.0921  0.1290  
2             2981.48   57.480   1.00  0.1318  0.1171  
3              128.95    1.423  45.30  0.0228  0.0147  
4              883.83    4.284  15.87  0.0336  

In [23]:
# Cap ROAS at 30x to keep it realistic
fact_marketing_spend['roas'] = fact_marketing_spend['roas'].clip(upper=30.0)

# Also cap attributed_revenue accordingly
fact_marketing_spend['attributed_revenue'] = (
    fact_marketing_spend['roas'] * fact_marketing_spend['spend']
).round(2)

print("✓ ROAS capped. Updated channel summary:")
print(fact_marketing_spend.groupby('channel')[['spend','attributed_revenue','roas','cac']].mean().round(2))

✓ ROAS capped. Updated channel summary:
             spend  attributed_revenue   roas    cac
channel                                             
Email        76.93             2307.64  29.99   0.81
Google      483.99             9592.73  19.89   5.75
Influencer  191.21              424.09   2.19  40.43
Instagram   291.46             1233.41   4.19  23.12


In [24]:
# ── Save fact_marketing_spend ──────────────────────────────────────────────
fact_marketing_spend.to_csv('fact_marketing_spend.csv', index=False)
print("✓ fact_marketing_spend.csv saved:", fact_marketing_spend.shape)

# ── Save fact_transactions ─────────────────────────────────────────────────
fact_transactions.to_csv('fact_transactions.csv', index=False)
print("✓ fact_transactions.csv saved:", fact_transactions.shape)

# ── Build dim_date ─────────────────────────────────────────────────────────
dim_date = pd.DataFrame({'date': pd.date_range('2017-01-01', '2018-08-31', freq='D')})

dim_date['year']        = dim_date['date'].dt.year
dim_date['month']       = dim_date['date'].dt.month
dim_date['month_name']  = dim_date['date'].dt.strftime('%B')
dim_date['quarter']     = dim_date['date'].dt.quarter
dim_date['week']        = dim_date['date'].dt.isocalendar().week.astype(int)
dim_date['day_of_week'] = dim_date['date'].dt.dayofweek        # 0=Monday
dim_date['day_name']    = dim_date['date'].dt.strftime('%A')
dim_date['is_weekend']  = dim_date['day_of_week'] >= 5

# Flag Nov-Dec as peak season
dim_date['is_peak_season'] = dim_date['month'].isin([11, 12])

dim_date.to_csv('dim_date.csv', index=False)
print("✓ dim_date.csv saved:", dim_date.shape)
print(dim_date.head(5))

✓ fact_marketing_spend.csv saved: (2432, 11)
✓ fact_transactions.csv saved: (97006, 30)
✓ dim_date.csv saved: (608, 10)
        date  year  month month_name  quarter  week  day_of_week   day_name  \
0 2017-01-01  2017      1    January        1    52            6     Sunday   
1 2017-01-02  2017      1    January        1     1            0     Monday   
2 2017-01-03  2017      1    January        1     1            1    Tuesday   
3 2017-01-04  2017      1    January        1     1            2  Wednesday   
4 2017-01-05  2017      1    January        1     1            3   Thursday   

   is_weekend  is_peak_season  
0        True           False  
1       False           False  
2       False           False  
3       False           False  
4       False           False  


In [25]:
import os
print(os.getcwd())

/Users/Tanya/Retail_Python_PowerBI


In [26]:
print(os.listdir())

['fact_marketing_spend.csv', 'dim_customers.csv', 'dim_customers_final.csv', 'Olist.ipynb', 'dim_date.csv', '.ipynb_checkpoints', 'bridge_customer_attribution.csv', 'fact_transactions.csv']


In [27]:
!pip install sqlalchemy psycopg2-binary

In [28]:
import pandas as pd
import numpy as np

# ── 0. LOAD DATA ───────────────────────────────────────────────────────────
# We load your saved files to ensure variables are defined
fact_transactions = pd.read_csv('fact_transactions.csv')
# Note: Ensure you have your RFM table ready as 'rfm_df' or load it here
# if you saved it. If not, we'll use the customer IDs from transactions.
unique_customers = fact_transactions['customer_unique_id'].unique()

print(f"Loaded {len(fact_transactions)} transactions for {len(unique_customers)} customers.")

# ── 1. MULTI-TOUCH ATTRIBUTION (MTA) BRIDGE ────────────────────────────────
# In a real brand, customers touch multiple channels. 
# We simulate a "Linear Attribution" model where credit is split equally.



channels = ['Instagram', 'Google', 'Email', 'Influencer']
np.random.seed(42)

touchpoint_data = []
for cust in unique_customers:
    # Most customers (70%) have 1 touchpoint, some have 2 or 3
    num_touches = np.random.choice([1, 2, 3], p=[0.7, 0.2, 0.1])
    selected_channels = np.random.choice(channels, num_touches, replace=True)
    
    for i, ch in enumerate(selected_channels):
        touchpoint_data.append({
            'customer_unique_id': cust,
            'channel': ch,
            'touchpoint_order': i + 1,
            'attribution_weight': 1.0 / num_touches  # Linear logic
        })

df_attribution_bridge = pd.DataFrame(touchpoint_data)

# ── 2. ENGINEER LTV & PAYBACK PERIOD ───────────────────────────────────────
# LTV (Lifetime Value) = Total Revenue * a multiplier (predicting future spend)
# Payback Period = How many days until the customer's gross margin covers their CAC

customer_stats = fact_transactions.groupby('customer_unique_id').agg({
    'total_revenue': 'sum',
    'order_id': 'nunique'
}).reset_index()

# Simulation logic for LTV: 
# Champions might spend 2x their initial value, while 'At Risk' spend 1x.
customer_stats['ltv_estimate'] = (customer_stats['total_revenue'] * 1.4).round(2)

# Payback Period Calculation:
# Let's assume an average CAC of $25. 
avg_cac = 25
customer_stats['payback_period_days'] = (avg_cac / (customer_stats['total_revenue'] / 365)).clip(upper=365).round(0)

# ── 3. EXPORT FINAL CLEAN TABLES ──────────────────────────────────────────
df_attribution_bridge.to_csv('bridge_customer_attribution.csv', index=False)
customer_stats.to_csv('dim_customers_final.csv', index=False)

print("✅ Phase 1 Successfully Finished!")
print(f"Generated {len(df_attribution_bridge)} attribution touchpoints.")
print(f"Calculated LTV metrics for {len(customer_stats)} customers.")

Loaded 97006 transactions for 93357 customers.
✅ Phase 1 Successfully Finished!
Generated 130801 attribution touchpoints.
Calculated LTV metrics for 93357 customers.


In [29]:
import pandas as pd
import numpy as np

# Load existing data
fact_transactions = pd.read_csv('fact_transactions.csv')
dim_customers_raw = pd.read_csv('dim_customers.csv')

# 1. Generate the Attribution Bridge
unique_customers = fact_transactions['customer_unique_id'].unique()
channels = ['Instagram', 'Google', 'Email', 'Influencer']
np.random.seed(42)

bridge_data = []
for cust in unique_customers:
    num_touches = np.random.choice([1, 2, 3], p=[0.7, 0.2, 0.1])
    selected = np.random.choice(channels, num_touches, replace=True)
    for i, ch in enumerate(selected):
        bridge_data.append({
            'customer_unique_id': cust,
            'channel': ch,
            'attribution_weight': 1.0 / num_touches
        })

df_bridge = pd.DataFrame(bridge_data)

# 2. Add LTV and Payback to Customers
cust_metrics = fact_transactions.groupby('customer_unique_id')['total_revenue'].sum().reset_index()
dim_customers_final = dim_customers_raw.merge(cust_metrics, on='customer_unique_id', how='left')

# Advanced Engineering formulas
dim_customers_final['ltv_estimate'] = (dim_customers_final['total_revenue'] * 1.4).fillna(0)
dim_customers_final['payback_period_days'] = (25 / (dim_customers_final['total_revenue'] / 365)).clip(upper=365).fillna(365)

# 3. Save the new versions
df_bridge.to_csv('bridge_customer_attribution.csv', index=False)
dim_customers_final.to_csv('dim_customers_final.csv', index=False)

print("✅ Files updated. You now have the 'Bridge' and 'Final Dim' ready.")

✅ Files updated. You now have the 'Bridge' and 'Final Dim' ready.


In [62]:
from sqlalchemy import create_engine, text

# 1. Update these to match what worked in your Terminal check
USER = 'postgres'
PASS = 'asmat786' # Leave as '' if no password worked in terminal
HOST = 'localhost'
PORT = '5432'
DB   = 'olist_marketing'

engine = create_engine(f'postgresql://{USER}:{PASS}@{HOST}:{PORT}/{DB}')

# 2. Test the connection immediately
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT 1"))
        print("🚀 CONNECTION SUCCESSFUL!")
except Exception as e:
    print(f"❌ CONNECTION FAILED: {e}")

# 3. Rename columns to match your SQL exactly BEFORE uploading
fact_marketing_spend = fact_marketing_spend.rename(columns={'attributed_revenue': 'revenue_attributed'})

# 4. Push the data (Using 'replace' to ensure the bridge is built)
print("Uploading data...")
fact_transactions.to_sql('fact_transactions', engine, if_exists='replace', index=False)
fact_marketing_spend.to_sql('fact_marketing_spend', engine, if_exists='replace', index=False)
dim_customers_final.to_sql('dim_customers', engine, if_exists='replace', index=False)
dim_date.to_sql('dim_date', engine, if_exists='replace', index=False)
df_bridge.to_sql('bridge_customer_attribution', engine, if_exists='replace', index=False)

print("✅ ALL DATA PUSHED TO POSTGRES!")

🚀 CONNECTION SUCCESSFUL!
Uploading data...
✅ ALL DATA PUSHED TO POSTGRES!


In [7]:
import pandas as pd

# Load files
df_customers = pd.read_csv('dim_customers_final.csv')
df_marketing = pd.read_csv('fact_marketing_spend.csv')
df_transactions = pd.read_csv('fact_transactions.csv')

print("=== GROUND TRUTH KPI VERIFICATION ===\n")

# 1. True Average Customer LTV
avg_ltv = df_customers['ltv_estimate'].mean()
print(f"1. True Average Customer LTV: ${avg_ltv:.2f}")

# 2. True Total Revenue
total_rev = df_transactions['total_revenue'].sum()
print(f"2. True Total Revenue:         ${total_rev:,.2f}")

# 3. Dynamic search for the spend column to prevent KeyError
spend_col = [col for col in df_marketing.columns if 'spend' in col.lower() or 'cost' in col.lower()]

if spend_col:
    actual_spend_col = spend_col[0]
    total_spend = df_marketing[actual_spend_col].sum()
    global_roas = total_rev / total_spend
    
    print(f"\nFound Spend Column Name:      '{actual_spend_col}'")
    print(f"Total Marketing Spend:        ${total_spend:,.2f}")
    print(f"3. True Global ROAS:          {global_roas:.2f}x  (or {global_roas*100:.2f}%)")
else:
    print("\n[!] Error: Could not find a spend or cost column in fact_marketing_spend.csv")
    print("Available columns are:", list(df_marketing.columns))

=== GROUND TRUTH KPI VERIFICATION ===

1. True Average Customer LTV: $232.28
2. True Total Revenue:         $15,489,522.09

Found Spend Column Name:      'spend'
Total Marketing Spend:        $634,503.28
3. True Global ROAS:          24.41x  (or 2441.20%)
